# Question 1 — Ethena loop mechanics, funding source, max leverage, ROE

**Question:** How does the Ethena loop work, and who is funding it? Trace the yield to its ultimate economic source. What is the maximum leverage achievable under current E-mode parameters, and the resulting ROE?

Full discovery methodology lives in `00_setup_data_sourcing.ipynb`; this notebook uses the result directly.

## 1.1 — How Ethena generates USDe's yield

Ethena issues USDe, backed by a portfolio of yield-generating strategies. Most recent disclosed breakdown ([Ethena reserve diversification announcement, 2026-04-07](https://unchainedcrypto.com/ethena-overhauls-usde-reserves-with-institutional-lending-and-real-world-assets/)):

| Source | Share / status |
|---|---|
| Delta-neutral basis trade (perpetual futures funding) | **11%** of backing |
| Stablecoin reserves + DeFi lending positions | remainder (~89%, exact split not disclosed) |
| Institutional lending (Anchorage Digital, Maple Institutional, Coinbase Asset Management) | expanding, no % disclosed yet |
| Real-world assets (tokenized T-bills, CLOs, investment-grade corporate bonds, structured credit) | expanding, no % disclosed yet |
| Equity/commodity basis trades, prime lending to trading firms | newly added categories, no % disclosed |
| Reserve Fund (loss-absorbing buffer) | ~$80M+ (early 2026) |

**Flagged assumption:** most recent *disclosed* composition (April 2026), not a live daily breakdown. Notably, perp funding — once described as the "primary engine" — is now a minority (11%) of backing.

**Basis trade:** Ethena holds a long spot/staking position hedged by an equal-notional short perp. In a structurally long-biased perp market, longs pay funding to shorts, so Ethena (as the short) collects it — this flips negative in bearish/crowded-short regimes.

**sUSDe:** USDe holders stake into an ERC-4626 vault for sUSDe; as the vault's balance grows from yield, each share redeems for more USDe — yield as a rising exchange rate, not a rebase.

## 1.2 — The loop

Deposit sUSDe (E-mode) → borrow USDe → stake it into sUSDe → redeposit → repeat. Each pass adds a shrinking increment, converging to a fixed max position (1.4), not growing forever.

At the case rates (sUSDe 4.0%, USDe borrow 3.2%; on-chain confirms 3.298%), the looper earns 4.0% on the *full* leveraged stack while paying 3.2% only on the *borrowed* portion — leverage amplifies the positive spread.

## 1.3 — Tracing the yield to its ultimate economic source

Aave is only the leverage venue — it doesn't set the yield. Given the portfolio mix, the yield traces to **two** distinct sources:

1. **Leveraged perp demand** (11% of backing): paid by leveraged longs on ETH perps. Not free money — a bet that longs keep paying to stay long, which can go negative in bear/crowded-short regimes.
2. **The cost of USD credit** (the larger, growing remainder — reserves, DeFi lending, institutional loans, T-bills/RWA): anchored to the Fed's policy rate, currently **3.50–3.75%** ([FOMC, held since 2026-06-17](https://www.federalreserve.gov/newsevents/pressreleases/monetary20260617a.htm)). This portion is a credit spread over the policy rate, paid by institutional borrowers and, for the T-bill leg, the Treasury.

**Why it matters:** as backing diversifies away from perp funding, sUSDe's yield becomes less a crypto-leverage-cycle bet and more a spread over Fed funds. At 13.2% looped ROE (1.5) vs. 3.50–3.75% Fed funds, that gap is what draws looping activity.

## 1.4 — E-mode parameters

Read directly from the Pool contract at **block 25,682,519** (category-32 discovery methodology in `00_setup_data_sourcing.ipynb`). Contract: `0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2` — [Read as Proxy](https://etherscan.io/address/0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2#readProxyContract), `getEModeCategoryCollateralConfig(32)`.

In [28]:
from web3 import Web3
import datetime, json

RPC = "https://eth.drpc.org"  # archive endpoint -- retains state at our pinned block indefinitely
w3 = Web3(Web3.HTTPProvider(RPC))
assert w3.is_connected(), "RPC not reachable"

POOL = w3.to_checksum_address("0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2")
BLOCK_NUMBER = 25682519
BLOCK_TS = datetime.datetime.utcfromtimestamp(1785858143)

ABI = json.loads('''[
 {"name":"getEModeCategoryCollateralConfig","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],
  "outputs":[{"type":"tuple","components":[{"name":"ltv","type":"uint16"},
   {"name":"liquidationThreshold","type":"uint16"},{"name":"liquidationBonus","type":"uint16"}]}]},
 {"name":"getEModeCategoryLabel","type":"function","stateMutability":"view",
  "inputs":[{"name":"id","type":"uint8"}],"outputs":[{"type":"string"}]}
]''')
pool = w3.eth.contract(address=POOL, abi=ABI)

EMODE_ID = 32
cfg = pool.functions.getEModeCategoryCollateralConfig(EMODE_ID).call(block_identifier=BLOCK_NUMBER)
label = pool.functions.getEModeCategoryLabel(EMODE_ID).call(block_identifier=BLOCK_NUMBER)

LTV = cfg[0] / 10000        # bps of 1.0 -> fraction
LT  = cfg[1] / 10000
LIQ_BONUS = (cfg[2] - 10000) / 10000

print(f"Category {EMODE_ID} ({label}) at block {BLOCK_NUMBER}, {BLOCK_TS} UTC:")
print(f"  LTV                 = {LTV:.2%}")
print(f"  Liquidation Threshold = {LT:.2%}")
print(f"  Liquidation Bonus   = {LIQ_BONUS:.2%}")

Category 32 (PTsUSDe5FEB/USDe) at block 25682519, 2026-08-04 15:42:23 UTC:
  LTV                 = 92.00%
  Liquidation Threshold = 94.00%
  Liquidation Bonus   = 2.00%


**What these mean:**

- **LTV 92%:** max borrow against collateral at open — $92 per $100 of sUSDe. Sets the leverage ceiling (1.5).
- **Liquidation Threshold 94%:** collateral ratio at which a position becomes liquidatable. The 2-point LTV→LT gap is the buffer; opening at the LTV cap leaves none.
- **Liquidation Bonus 2%:** discount to the liquidator (haircut to the borrower) on liquidation.

## 1.5 — Maximum leverage and ROE

Looping to the LTV cap is a geometric series: each pass supplies collateral, borrows $LTV$ of it, restakes, and resupplies.

$$\text{Max leverage } L = \frac{1}{1 - LTV}$$

At convergence, collateral $C = E \cdot L$ and debt $D = E \cdot (L-1)$, so:

$$\text{ROE} = \frac{C \cdot y_{sUSDe} - D \cdot r_{USDe}}{E} = L \cdot y_{sUSDe} - (L-1) \cdot r_{USDe}$$

using the assignment's case inputs ($y_{sUSDe} = 4.0\%$, $r_{USDe} = 3.2\%$).

In [29]:
EQUITY = 1000.0
USDE_BORROW_APY = 0.032
SUSDE_NET_YIELD = 0.040

L = 1 / (1 - LTV)
C = EQUITY * L
D = EQUITY * (L - 1)
annual_pnl = C * SUSDE_NET_YIELD - D * USDE_BORROW_APY
roe = annual_pnl / EQUITY

print(f"Max leverage           = 1 / (1 - {LTV:.2f}) = {L:.2f}x")
print(f"Collateral (sUSDe)      = ${C:,.2f}")
print(f"Debt (USDe)             = ${D:,.2f}")
print(f"Net annual P&L on $1,000 = ${annual_pnl:,.2f}")
print(f"ROE                     = {roe:.2%}")
print(f"\nFed Funds Rate (upper)  = 3.75%  -- ROE exceeds this by {roe - 0.0375:.2%}, which is the spread that attracts looping activity.")

Max leverage           = 1 / (1 - 0.92) = 12.50x
Collateral (sUSDe)      = $12,500.00
Debt (USDe)             = $11,500.00
Net annual P&L on $1,000 = $132.00
ROE                     = 13.20%

Fed Funds Rate (upper)  = 3.75%  -- ROE exceeds this by 9.45%, which is the spread that attracts looping activity.


## 1.6 — Summary

- **Mechanism:** deposit sUSDe → borrow USDe in E-mode (category 32) → mint more sUSDe → redeposit, converging to $L = 1/(1-LTV)$.
- **E-mode parameters:** LTV 92%, Liquidation Threshold 94%, Liquidation Bonus 2% — verified against the Pool contract at block 25,682,519.
- **Max leverage: 12.50x.** ROE at max leverage on $1,000: **13.20%** ($132/yr) — well above the Fed Funds Rate (3.50–3.75%), the gap that attracts looping activity.
- **Not a free-standing ceiling:** 92% LTV is zero safety margin — a rational looper runs below it. This is a theoretical maximum, not what's typically deployed.
- **Yield source:** not Aave, and not solely crypto-native funding — increasingly a credit spread anchored to Fed policy as Ethena's backing diversifies (11% perp funding, majority now DeFi lending/institutional loans/RWA). Aave is a leverage venue on a yield it doesn't set and isn't a counterparty to.